### PulmoTriage : Vision Model (chest X-ray)

Predicts the 110 findings from the X-rays with a frozen CheXNet and a small classifier.

#### Input files

In [ ]:
import glob, json

def find_file(name):
    hits = sorted(glob.glob(f'/kaggle/input/**/{name}', recursive=True))
    return hits[0] if hits else None

SPLITS_CSV   = find_file('splits.csv')
LABELS_CSV   = find_file('nlp_dataset.csv')
CLASSES_JSON = find_file('mlb_classes.json')
CHEXNET_H5   = find_file('brucechou1983_CheXNet_Keras_0.3.0_weights.h5')
PROJ_CSV     = find_file('indiana_projections.csv')
OUT_DIR = '/kaggle/working'
print(SPLITS_CSV, LABELS_CSV, CLASSES_JSON, CHEXNET_H5, PROJ_CSV, sep='\n')
assert all([SPLITS_CSV, LABELS_CSV, CLASSES_JSON, CHEXNET_H5, PROJ_CSV]), 'an input is missing'

#### Imports

In [ ]:
import os, sys, shutil, importlib
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import roc_auc_score, f1_score, roc_curve
from sklearn.preprocessing import StandardScaler

SEED = 42
keras.utils.set_random_seed(SEED)

### Vision module (`pulmo_vision.py`)

The same code reads the images here and in the app, so both always get the same features.

In [ ]:
%%writefile pulmo_vision.py
import io
import json
import os

import numpy as np
import tensorflow as tf
from tensorflow import keras

IMG_SIZE = 224
MAX_IMAGES = 5

def decode_image(data):
    image = tf.io.decode_image(data, channels=1, expand_animations=False)
    image = tf.cast(image, tf.float32)
    image = tf.image.resize_with_pad(image, IMG_SIZE, IMG_SIZE)
    return image / 255.0

def load_image(source):
    if isinstance(source, (str, os.PathLike)):
        data = tf.io.read_file(str(source))
    elif isinstance(source, (bytes, bytearray)):
        data = tf.constant(bytes(source))
    elif hasattr(source, "getvalue"):
        data = tf.constant(source.getvalue())
    else:
        buf = io.BytesIO()
        source.save(buf, format="PNG")
        data = tf.constant(buf.getvalue())
    return decode_image(data)

def load_chexnet(weights_path):
    import h5py
    model = keras.applications.DenseNet121(include_top=False, weights=None, input_shape=(IMG_SIZE, IMG_SIZE, 3))
    model.load_weights(weights_path, by_name=True, skip_mismatch=True)
    with h5py.File(weights_path, "r") as f:
        model.get_layer("conv1_conv").set_weights([f["conv1/conv/conv1/conv/kernel:0"][()]])
        bn = f["conv1/bn/conv1/bn"]
        model.get_layer("conv1_bn").set_weights([bn["gamma:0"][()], bn["beta:0"][()],
                                                 bn["moving_mean:0"][()], bn["moving_variance:0"][()]])
        assert np.allclose(model.get_layer("conv1_conv").get_weights()[0], f["conv1/conv/conv1/conv/kernel:0"][()])
        last = f["bn/bn"]
        assert np.allclose(model.get_layer("bn").get_weights()[0], last["gamma:0"][()]), "CheXNet weights not loaded"
    model.trainable = False
    return model

class VisionModel:
    def __init__(self, model_dir="vision_model"):
        self.dir = model_dir
        cfg = json.load(open(os.path.join(model_dir, "config.json")))
        self.classes = cfg["classes"]
        self.thr_image = np.array(cfg["thresholds_image"])
        self.thr_fusion = np.array(cfg["thresholds_fusion"])
        self.fusion_method = cfg["fusion_method"]
        self.late_weight = cfg["late_fusion_image_weight"]
        self.urgent = [c for c in cfg["urgent_findings"] if c in self.classes]
        sc = np.load(os.path.join(model_dir, "scalers.npz"))
        self.img_mean, self.img_scale = sc["img_mean"], sc["img_scale"]
        self.txt_mean, self.txt_scale = sc["txt_mean"], sc["txt_scale"]
        self.image_model = keras.models.load_model(os.path.join(model_dir, "image_model.keras"))
        self.fusion_model = keras.models.load_model(os.path.join(model_dir, "fusion_model.keras"))
        self.text_only_model = keras.models.load_model(os.path.join(model_dir, "text_only_model.keras"))

        self.chexnet = load_chexnet(os.path.join(model_dir, cfg["chexnet_weights"]))

    def _conv(self, images):
        x = tf.stack([load_image(s) for s in images[:MAX_IMAGES]])
        return self.chexnet(tf.concat([x, x, x], axis=-1), training=False)

    def embed(self, images):
        return tf.reduce_mean(self._conv(images), axis=[1, 2]).numpy().mean(axis=0)

    def _scale_img(self, emb):
        return ((emb - self.img_mean) / self.img_scale).astype("float32")[None, :]

    def _scale_txt(self, emb, has_note):
        return (((emb - self.txt_mean) / self.txt_scale) * has_note).astype("float32")[None, :]

    def _result(self, p, thresholds, top_k=5):
        order = np.argsort(-p)
        item = lambda i: {"label": self.classes[i], "probability": round(float(p[i]), 4)}
        res = {"findings": [item(i) for i in order if p[i] > thresholds[i] and self.classes[i] != "normal"],
               "top": [item(i) for i in order if self.classes[i] != "normal"][:top_k],
               "probabilities": {c: float(v) for c, v in zip(self.classes, p)}}
        if "normal" in self.classes:
            res["p_normal"] = round(float(p[self.classes.index("normal")]), 4)
        return res

    def predict_image(self, images, with_region=True):
        p = self.image_model.predict(self._scale_img(self.embed(images)), verbose=0)[0]
        res = self._result(p, self.thr_image)
        if with_region:
            res["region"] = self.heatmap(images[0])[1]
        return res

    def predict_fusion(self, images, note=None, text_model=None):
        has_note = 0.0
        txt = np.zeros_like(self.txt_mean)
        if note and str(note).strip() and text_model is not None:
            txt = text_model.embed([note])[0]
            has_note = float(np.abs(txt).sum() > 0)
        x_img = self._scale_img(self.embed(images))
        x_txt = self._scale_txt(txt, has_note)
        note_col = np.array([[has_note]], dtype="float32")
        if self.fusion_method == "early":
            p = self.fusion_model.predict(np.hstack([x_img, x_txt, note_col]), verbose=0)[0]
        else:
            p_img = self.image_model.predict(x_img, verbose=0)[0]
            p_txt = self.text_only_model.predict(np.hstack([x_txt, note_col]), verbose=0)[0]
            p = self.late_weight * p_img + (1 - self.late_weight) * p_txt if has_note else p_img
        res = self._result(p, self.thr_fusion)
        res["used_note"] = bool(has_note)
        return res

    def priority(self, result):
        flagged = [d["label"] for d in result["findings"]]
        if any(f in self.urgent for f in flagged):
            return "high"
        if flagged:
            return "medium"
        return "low"

    def heatmap(self, image, label=None):
        x = load_image(image)
        if label is None:
            p = self.image_model.predict(self._scale_img(self.embed([image])), verbose=0)[0]
            p = np.where(np.array(self.classes) == "normal", -1, p)
            k = int(np.argmax(p))
        else:
            k = self.classes.index(label)
        x3 = tf.concat([x, x, x], axis=-1)[None]
        mean, scale = tf.constant(self.img_mean, tf.float32), tf.constant(self.img_scale, tf.float32)
        with tf.GradientTape() as tape:
            conv = self.chexnet(x3, training=False)
            tape.watch(conv)
            pooled = (tf.reduce_mean(conv, axis=[1, 2]) - mean) / scale
            score = self.image_model(pooled, training=False)[0, k]
        grads = tape.gradient(score, conv)[0]
        weights = tf.reduce_mean(grads, axis=[0, 1])
        cam = tf.nn.relu(tf.reduce_sum(conv[0] * weights, axis=-1)).numpy()
        cam = cam / (cam.max() + 1e-8)
        cam = tf.image.resize(cam[..., None], (IMG_SIZE, IMG_SIZE)).numpy()[..., 0]

        r, c = np.unravel_index(np.argmax(cam), cam.shape)
        zone = ["upper", "middle", "lower"][min(2, int(3 * r / IMG_SIZE))]
        side = "right" if c < IMG_SIZE / 2 else "left"
        region = f"{side} {zone} zone"

        gray = np.repeat(x.numpy(), 3, axis=-1)
        heat = np.stack([cam, np.zeros_like(cam), np.zeros_like(cam)], axis=-1)
        overlay = np.clip(0.65 * gray + 0.5 * heat, 0, 1)
        return (overlay * 255).astype("uint8"), region

In [ ]:
sys.path.insert(0, '.')
import pulmo_vision; importlib.reload(pulmo_vision)
chexnet = pulmo_vision.load_chexnet(CHEXNET_H5)
print('CheXNet loaded:', len(chexnet.weights), 'weight tensors')

### Image features

Every X-ray goes through CheXNet; a patient's features are the average over their images (up to 5).

#### Image files per patient

In [ ]:
splits = pd.read_csv(SPLITS_CSV)
IMAGE_DIR = os.path.join(os.path.dirname(PROJ_CSV), 'images', 'images_normalized')
proj = pd.read_csv(PROJ_CSV)
proj = proj[proj.uid.isin(splits.uid)].copy()
proj['path'] = [os.path.join(IMAGE_DIR, f) for f in proj.filename]
proj = proj[proj.path.map(os.path.exists)]
proj = proj.groupby('uid').head(pulmo_vision.MAX_IMAGES).reset_index(drop=True)
print(len(proj), 'images |', proj.uid.nunique(), 'of', len(splits), 'patients')

#### Encode every image with CheXNet

In [ ]:
def read(path):
    x = pulmo_vision.decode_image(tf.io.read_file(path))
    return tf.concat([x, x, x], axis=-1)

encoder = keras.Sequential([chexnet, layers.GlobalAveragePooling2D()])
dataset = tf.data.Dataset.from_tensor_slices(proj.path.values).map(read).batch(16)
features = encoder.predict(dataset, verbose=1)

#### Average per patient

In [ ]:
IMG_COLS = [f'img_{i}' for i in range(features.shape[1])]
img = pd.DataFrame(features, columns=IMG_COLS)
img.insert(0, 'uid', proj.uid.values)
img = img.groupby('uid', sort=False).mean().reset_index()
img = splits.merge(img, on='uid')
img.to_csv(f'{OUT_DIR}/image_embeddings.csv', index=False, float_format='%.6g')
print('image features:', img.shape)

### Data

#### Load labels

In [ ]:
CLASSES = json.load(open(CLASSES_JSON))
labels = pd.read_csv(LABELS_CSV, usecols=['uid'] + CLASSES).drop_duplicates('uid')
df = img.merge(labels, on='uid')
df.split.value_counts()

#### Train / val / test arrays

In [ ]:
train, val, test = df[df.split == 'train'], df[df.split == 'val'], df[df.split == 'test']
img_scaler = StandardScaler().fit(train[IMG_COLS])

def get_xy(d):
    return img_scaler.transform(d[IMG_COLS]).astype('float32'), d[CLASSES].values.astype('float32')

img_tr, y_tr = get_xy(train)
img_va, y_va = get_xy(val)
img_te, y_te = get_xy(test)
print('train', len(y_tr), '| val', len(y_va), '| test', len(y_te))

### Metric

Macro ROC-AUC: the AUC of each finding, averaged, so rare findings count as much as common ones.

In [ ]:
def macro_auc(y, p, min_pos=5):
    keep = [k for k in range(y.shape[1]) if min_pos <= y[:, k].sum() < len(y)]
    return np.mean([roc_auc_score(y[:, k], p[:, k]) for k in keep])

def confidence_interval(y, p, n=200):
    rng = np.random.default_rng(SEED)
    scores = []
    for _ in range(n):
        idx = rng.integers(0, len(y), len(y))
        scores.append(macro_auc(y[idx], p[idx], min_pos=1))
    return np.percentile(scores, [2.5, 97.5])

### Model

A small network on the CheXNet features; training stops when the validation loss stops improving.

#### Network

In [ ]:
def build_model(n_inputs):
    model = keras.Sequential([
        layers.Input(shape=(n_inputs,)),
        layers.Dropout(0.2),
        layers.Dense(512, activation='relu'),
        layers.Dropout(0.4),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.4),
        layers.Dense(len(CLASSES), activation='sigmoid'),
    ])
    model.compile(optimizer=keras.optimizers.Adam(1e-3), loss='binary_crossentropy')
    return model

#### Train

In [ ]:
image_model = build_model(img_tr.shape[1])
stop = keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
history = image_model.fit(img_tr, y_tr, validation_data=(img_va, y_va),
                          epochs=60, batch_size=64, callbacks=[stop], verbose=0)
p_img_va = image_model.predict(img_va, verbose=0)
print(f'stopped at epoch {int(np.argmin(history.history["val_loss"])) + 1} | val macro ROC-AUC {macro_auc(y_va, p_img_va, 1):.4f}')

#### Learning curve

In [ ]:
plt.plot(history.history['loss'], label='train')
plt.plot(history.history['val_loss'], label='val')
plt.xlabel('epoch'); plt.ylabel('loss'); plt.legend(); plt.title('Vision model loss')
plt.savefig(f'{OUT_DIR}/vision_learning_curve.png', dpi=120); plt.show()

### Decision thresholds

Each finding gets its own threshold, chosen on the validation set.

In [ ]:
def best_thresholds(y, p):
    thresholds = []
    for k in range(y.shape[1]):
        best_f1, best_t = 0, 1.01
        if y[:, k].sum() > 0:
            for t in np.unique(np.round(p[:, k], 3)):
                flag = p[:, k] > t
                if flag.mean() > 2 * y[:, k].mean() + 1 / len(y):
                    continue
                f1 = f1_score(y[:, k], flag, zero_division=0)
                if f1 > best_f1:
                    best_f1, best_t = f1, float(t)
        thresholds.append(best_t)
    return np.array(thresholds)

In [ ]:
thr_image = best_thresholds(y_va, p_img_va)
print('findings the image model can flag:', int((thr_image < 1.01).sum()))

### Test set evaluation

#### Overall

In [ ]:
p_img = image_model.predict(img_te, verbose=0)
low, high = confidence_interval(y_te, p_img)
print(f'Image only · test macro ROC-AUC {macro_auc(y_te, p_img):.3f} (95% CI {low:.3f}–{high:.3f})')

#### Per finding

In [ ]:
rows = []
for k, c in enumerate(CLASSES):
    if y_te[:, k].sum() >= 5:
        rows.append({'finding': c, 'test cases': int(y_te[:, k].sum()), 'AUC': round(roc_auc_score(y_te[:, k], p_img[:, k]), 3)})
per_finding = pd.DataFrame(rows).set_index('finding').sort_values('AUC', ascending=False)
per_finding.to_csv(f'{OUT_DIR}/vision_per_finding.csv')
per_finding.head(15)

#### ROC curves of important findings

In [ ]:
important = ['cardiomegaly', 'pleural effusion', 'opacity', 'pulmonary atelectasis', 'nodule', 'consolidation', 'normal']
plt.figure(figsize=(6, 6))
for c in [c for c in important if c in per_finding.index]:
    k = CLASSES.index(c)
    fpr, tpr, _ = roc_curve(y_te[:, k], p_img[:, k])
    plt.plot(fpr, tpr, label=f'{c} ({roc_auc_score(y_te[:, k], p_img[:, k]):.2f})')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('false positive rate'); plt.ylabel('true positive rate'); plt.legend(fontsize=8); plt.title('Vision model ROC curves')
plt.savefig(f'{OUT_DIR}/vision_roc_curves.png', dpi=120); plt.show()

### Grad-CAM heatmaps

Where the model looks on a few test X-rays.

In [ ]:
class ImageModel:
    def __init__(self):
        self.chexnet, self.image_model, self.classes = chexnet, image_model, CLASSES
        self.img_mean, self.img_scale = img_scaler.mean_, img_scaler.scale_

In [ ]:
gradcam = ImageModel()
p_rank = np.where(np.array(CLASSES) == 'normal', -1, p_img)
top = np.argsort(-p_rank.max(axis=1))[:4]

fig, axes = plt.subplots(1, 4, figsize=(16, 4.5))
for ax, i in zip(axes, top):
    uid, k = test.uid.values[i], int(p_rank[i].argmax())
    file = proj.loc[proj.uid == uid, 'path'].iloc[0]
    overlay, region = pulmo_vision.VisionModel.heatmap(gradcam, file, label=CLASSES[k])
    ax.imshow(overlay); ax.axis('off')
    ax.set_title(f'{CLASSES[k]} {p_img[i, k]:.0%} · {region}', fontsize=9)
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/vision_heatmaps.png', dpi=120); plt.show()

### Save

In [ ]:
image_model.save(f'{OUT_DIR}/image_model.keras')
np.savez(f'{OUT_DIR}/image_scaler.npz', mean=img_scaler.mean_, scale=img_scaler.scale_)
json.dump({'classes': CLASSES, 'thresholds_image': thr_image.tolist()}, open(f'{OUT_DIR}/vision_config.json', 'w'))
pd.DataFrame(p_img, columns=[f'p_{c}' for c in CLASSES]).assign(uid=test.uid.values).to_csv(f'{OUT_DIR}/vision_test_predictions.csv', index=False)
print('Saved image_model.keras, image_scaler.npz, vision_config.json, image_embeddings.csv, vision_test_predictions.csv')